In [ ]:
# How `F.linear` works

Expert habit: **print shapes → compute two ways → assert they match**.

Router context: `F.linear(x, W)` with `W` shaped `(n_experts, hidden)` scores every token against every expert.

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# one token, hidden=4; weight: 3 experts × 4 hidden
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])          # (1, 4)
W = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # expert 0: pick x0
    [0.0, 1.0, 0.0, 0.0],   # expert 1: pick x1
    [0.25, 0.25, 0.25, 0.25],  # expert 2: average
])                                            # (3, 4)

y_linear = F.linear(x, W)                     # (1, 3)
y_matmul = x @ W.T                            # same math: y = x W^T

print("x       ", tuple(x.shape), x)
print("W       ", tuple(W.shape))
print("F.linear", tuple(y_linear.shape), y_linear)
print("x @ W.T ", tuple(y_matmul.shape), y_matmul)
assert torch.allclose(y_linear, y_matmul)
print("PASS: F.linear == x @ W.T")

x        (1, 4) tensor([[1., 2., 3., 4.]])
W        (3, 4)
F.linear (1, 3) tensor([[1.0000, 2.0000, 2.5000]])
x @ W.T  (1, 3) tensor([[1.0000, 2.0000, 2.5000]])
PASS: F.linear == x @ W.T


## Same op on a batch of tokens (MoE router shape)

Flatten `(B, S, H) → (B·S, H)`, then linear → `(B·S, n_experts)`.

In [2]:
B, S, H, N = 2, 4, 8, 6   # batch, seq, hidden, n_experts

hidden = torch.randn(B, S, H)
W_gate = torch.randn(N, H)                 # like TopkRouter.weight

flat = hidden.view(-1, H)                  # (8, 8)
logits = F.linear(flat, W_gate)            # (8, 6)

print("hidden ", tuple(hidden.shape))
print("flat   ", tuple(flat.shape))
print("W_gate ", tuple(W_gate.shape))
print("logits ", tuple(logits.shape), "← one score per expert, per token")
print("token0 scores:", logits[0])

# linear also works without flattening (applies on last dim)
logits_3d = F.linear(hidden, W_gate)       # (2, 4, 6)
assert torch.allclose(logits, logits_3d.view(-1, N))
print("PASS: flat linear matches 3D linear")

hidden  (2, 4, 8)
flat    (8, 8)
W_gate  (6, 8)
logits  (8, 6) ← one score per expert, per token
token0 scores: tensor([-1.2955,  1.0110, -1.1253, -4.9447, -4.5094,  0.8540])
PASS: flat linear matches 3D linear


## `nn.Linear` vs `F.linear`

Module stores the weight; functional takes the weight as an argument. Same math.

In [3]:
layer = torch.nn.Linear(H, N, bias=False)
layer.weight.data.copy_(W_gate)            # same W as above

y_mod = layer(flat)
y_fn  = F.linear(flat, layer.weight)

assert torch.allclose(y_mod, y_fn)
print("PASS: nn.Linear(x) == F.linear(x, layer.weight)")
print(y_mod.shape)

PASS: nn.Linear(x) == F.linear(x, layer.weight)
torch.Size([8, 6])


## What `nn.Parameter(torch.empty(...))` is

Router line:

```python
self.weight = nn.Parameter(torch.empty((n_routed_experts, hidden_size)))
```

Inspect: **shape → type → garbage values → after init → use in F.linear**.

In [4]:
n_routed_experts, hidden_size = 6, 8   # toy sizes

# 1) raw storage: uninitialized memory (garbage numbers — NOT zeros)
raw = torch.empty((n_routed_experts, hidden_size))
print("torch.empty shape:", tuple(raw.shape))
print("torch.empty sample row:", raw[0])
print("  ↑ look like random junk; empty() does NOT initialize")

# 2) wrap as nn.Parameter so optimizers / .parameters() can see it
weight = torch.nn.Parameter(raw)
print("\ntype(weight):", type(weight))
print("isinstance Parameter?", isinstance(weight, torch.nn.Parameter))
print("requires_grad:", weight.requires_grad)   # True by default for Parameter

# 3) real models almost always init after empty (GLM does this in _init_weights)
torch.nn.init.normal_(weight, mean=0.0, std=0.02)
print("\nafter normal_ init, row0:", weight[0].detach())

# 4) use it like the router: score tokens → experts
tokens = torch.randn(4, hidden_size)            # 4 tokens
logits = F.linear(tokens, weight)               # (4, 6)
print("\nF.linear(tokens, weight) shape:", tuple(logits.shape))
print("token0 expert scores:", logits[0].detach())

torch.empty shape: (6, 8)
torch.empty sample row: tensor([0., 0., 0., 0., 0., 0., 0., 0.])
  ↑ look like random junk; empty() does NOT initialize

type(weight): <class 'torch.nn.parameter.Parameter'>
isinstance Parameter? True
requires_grad: True

after normal_ init, row0: tensor([ 0.0193, -0.0198,  0.0060, -0.0021,  0.0200, -0.0100,  0.0152,  0.0124])

F.linear(tokens, weight) shape: (4, 6)
token0 expert scores: tensor([-0.0335,  0.0315, -0.0984,  0.0124, -0.0080, -0.0187])
